# Knowledge Tracing Evaluation
**BKT (Bayesian Knowledge Tracing) vs Elo Baseline**


In [ ]:
import sys, json
from pathlib import Path
import os
sys.path.insert(0, os.getcwd())

from tutor.curriculum_loader import CurriculumLoader
from tutor.adaptive import (
    BKTTracker, EloTracker, AdaptiveSelector,
    simulate_learner_replay, roc_auc, evaluate
)

SEED_PATH = "data/T3.1_Math_Tutor/curriculum_seed.json"
loader = CurriculumLoader(SEED_PATH)
print(f"Curriculum: {loader.summary()}")

NameError: name '__file__' is not defined

## Single Learner Simulation

In [ ]:
records = simulate_learner_replay(loader.items, n_steps=40, seed=0)
print(f"Simulated {len(records)} response events")
print(f"Sample records:")
for r in records[:5]:
    print(f"  skill={r['skill']:12s} diff={r['difficulty']}  "
          f"correct={r['correct']}  bkt_pred={r['bkt_pred']:.3f}  elo_pred={r['elo_pred']:.3f}")

## Multi-Learner AUC Evaluation

In [ ]:
N_LEARNERS = 20
results = evaluate(loader.items, n_learners=N_LEARNERS)
print("\n" + "="*45)
print("  KNOWLEDGE TRACING EVALUATION RESULTS")
print("="*45)
print(f"  Learners simulated : {results['n_learners']}")
print(f"  Total observations : {results['n_observations']}")
print(f"  BKT  AUC           : {results['BKT_AUC']:.4f}")
print(f"  Elo  AUC           : {results['Elo_AUC']:.4f}")
print(f"  BKT wins           : {results['BKT_wins']}")
print("="*45)

## Per-Skill BKT Trajectory

In [ ]:
from tutor.adaptive import BKTTracker, AdaptiveSelector
import random

bkt = BKTTracker()
elo = EloTracker()
selector = AdaptiveSelector(loader.items)
rng = random.Random(42)
true_mastery = {s: rng.uniform(0.1, 0.5) for s in ["counting","number_sense","addition","subtraction","word_problem"]}

trajectory = {s: [] for s in true_mastery}
for step in range(50):
    item = selector.select(bkt)
    if not item:
        break
    skill = item["skill"]
    p_correct = max(0.05, min(0.95, true_mastery[skill] + rng.gauss(0, 0.1)))
    correct = rng.random() < p_correct
    bkt.update(skill, correct)
    elo.update(skill, item["difficulty"], correct)
    trajectory[skill].append(bkt.p_know(skill))
    true_mastery[skill] = min(0.95, true_mastery[skill] + 0.015)

print("\nFinal BKT mastery estimates:")
for skill, vals in trajectory.items():
    if vals:
        bar = "█" * int(vals[-1] * 20)
        print(f"  {skill:14s}  {bar:20s}  {vals[-1]:.3f}")

## Save Results

In [ ]:
out = Path("reports")
out.mkdir(exist_ok=True)
with open(out / "kt_eval_results.json", "w") as f:
    json.dump(results, f, indent=2)
